In [8]:
# 01 패키지
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from pathlib import Path

In [9]:
# 02 랜덤 고정값
np.random.seed(42)
random.seed(42)

In [10]:
# 03 경로 설정
DATA_DIR = Path("../data")
OUTPUT_PATH = DATA_DIR / "13_profile_followers_demographics.csv"

In [11]:
# 04 profile_metrics 데이터 불러오기
profile_metrics = pd.read_csv(DATA_DIR / "12_profile_metrics.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
profile_metrics.columns = profile_metrics.columns.str.strip()

# 빈 문자열을 NaN으로 변환
profile_metrics = profile_metrics.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
profile_metrics = profile_metrics.dropna(how="all")

# influencer_id가 없는 행 제거
profile_metrics = profile_metrics.dropna(subset=["profile_metric_id"])

# 인덱스 재정렬
profile_metrics = profile_metrics.reset_index(drop=True)

# 데이터 확인
print(profile_metrics.shape)
profile_metrics.head()

(32, 11)


,profile_metric_id,client_id,profile_metric_at,profile_followers_count,profile_new_followers_count,profile_unfollowers_count,profile_reach_count,profile_impressions_count,profile_visit_count,profile_website_click_count,profile_website_click_rate
0,promet-0001,cli-0001,2026-05-14 00:00:00,574,7,4,3592,4870,43,1,2.33
1,promet-0002,cli-0001,2026-05-15 00:00:00,577,7,4,3671,4929,93,3,3.23
2,promet-0003,cli-0001,2026-05-16 00:00:00,581,7,3,2933,4052,37,2,5.41
3,promet-0004,cli-0001,2026-05-17 00:00:00,582,5,4,5343,6534,118,9,7.63
4,promet-0005,cli-0001,2026-05-18 00:00:00,581,2,3,1767,3167,16,1,6.25


In [12]:
# 05 profile_followers_demographics 데이터 생성

gender_list = ["male", "female"]
age_group_list = ["10대", "20대", "30대", "40대", "50대", "60대+"]

profile_followers_demographic_rows = []
profile_followers_demographic_id = 1

for _, metric in profile_metrics.iterrows():
    profile_metric_id = metric["profile_metric_id"]
    client_id = metric["client_id"]
    profile_metric_at = metric["profile_metric_at"]
    total_followers = int(metric["profile_followers_count"])

    # 성별/연령대 조합 생성
    segments = []

    for gender in gender_list:
        for age_group in age_group_list:
            segments.append({
                "profile_gender": gender,
                "profile_age_group": age_group
            })

    # 각 성별/연령대 조합에 배분할 비율 생성
    # 전체 비율의 합은 1이 됨
    ratios = np.random.dirichlet(np.ones(len(segments)))

    # 총 팔로워 수를 각 그룹에 배분
    # multinomial을 쓰면 그룹별 follower_count 합계가 total_followers와 정확히 일치함
    follower_counts = np.random.multinomial(total_followers, ratios)

    for segment, follower_count in zip(segments, follower_counts):
        follower_ratio = follower_count / total_followers if total_followers > 0 else 0

        profile_followers_demographic_rows.append({
            "profile_follower_id": f"profol-{profile_followers_demographic_id:04d}",
            "client_id": client_id,
            "profile_metric_id": profile_metric_id,
            "profile_metric_at": profile_metric_at,
            "profile_gender": segment["profile_gender"],
            "profile_age_group": segment["profile_age_group"],
            "profile_follower_count": follower_count,
            "profile_follower_ratio": round(follower_ratio, 4)*100
        })

        profile_followers_demographic_id += 1

profile_followers_demographics = pd.DataFrame(profile_followers_demographic_rows)

print(profile_followers_demographics.shape)
profile_followers_demographics.head()

(384, 8)


,profile_follower_id,client_id,profile_metric_id,profile_metric_at,profile_gender,profile_age_group,profile_follower_count,profile_follower_ratio
0,profol-0001,cli-0001,promet-0001,2026-05-14 00:00:00,male,10대,24,4.18
1,profol-0002,cli-0001,promet-0001,2026-05-14 00:00:00,male,20대,127,22.13
2,profol-0003,cli-0001,promet-0001,2026-05-14 00:00:00,male,30대,55,9.58
3,profol-0004,cli-0001,promet-0001,2026-05-14 00:00:00,male,40대,34,5.92
4,profol-0005,cli-0001,promet-0001,2026-05-14 00:00:00,male,50대,5,0.87


In [13]:
# 06 profile_metric_id별 demographics 팔로워 합계
demo_check = (
    profile_followers_demographics
    .groupby("profile_metric_id", as_index=False)["profile_follower_count"]
    .sum()
    .rename(columns={"profile_follower_count": "demographic_followers_sum"})
)

# 원본 profile_metrics의 팔로워 수
metric_check = profile_metrics[[
    "profile_metric_id",
    "profile_followers_count"
]].copy()

# 비교
validation = metric_check.merge(demo_check, on="profile_metric_id", how="left")

validation["difference"] = (
    validation["profile_followers_count"] 
    - validation["demographic_followers_sum"]
)

validation.head()

,profile_metric_id,profile_followers_count,demographic_followers_sum,difference
0,promet-0001,574,574,0
1,promet-0002,577,577,0
2,promet-0003,581,581,0
3,promet-0004,582,582,0
4,promet-0005,581,581,0


In [14]:
# 07 CSV 저장

profile_followers_demographics.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: ..\data\13_profile_followers_demographics.csv
